# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The Queue:**
This playbook prioritizes pages for manual editorial review. It uses the honest, grouped-split Random Forest model to flag content that is associated with high search visibility but poor click capture in the observed dataset.

**Archetype to Action Mapping:**

**The "Leaky Bucket" (High Impressions, Low CTR, Page 1 Rank):**

* **Action:** Review and refine the Meta Title and Description.

* **Reason Code:**`high_vis_poor_capture`. The model flags these because they already own valuable real estate but fail to convert it. Focus on tightening the promise statement to match search intent.

**The "Stale Winner" (High Past Traffic, High Age, Declining Impressions):**

* **Action:** Comprehensive content refresh.

* **Reason Code:** `stale_decay`. The FlyRank research paper observed that refreshing mature pages (365+ days old) was associated with a 3.2x higher health score compared to leaving them stale.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use:**
This queue provides directional decision-support for SEO editors and content managers. It is designed to answer the question, "Which URLs should our human team look at first this week to find the highest potential ROI?"

**Known Limits:**

* **Not Causal:** This model highlights observed patterns; it does not prove that changing a title will automatically increase traffic.

* **Intent Blindness:** The model relies on numerical metrics (impressions, ranks). It cannot read the search engine results page (SERP). It does not know if a keyword is a zero-click definition or a competitor's brand name.

* **Scale Limits:** The effect sizes measured in the underlying research apply to the specific portfolio analyzed. They should be treated as baseline expectations, not guaranteed future yields.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Review Rules:**
Before applying any changes to a flagged URL, a human editor MUST check the live SERP to answer:

1. Is this a zero-click search (e.g., a calculator or dictionary definition)?

2. Is the user clearly navigating to a specific brand portal that we do not own?

3. Is our page heavily mismatched to the actual intent of the query?

**The No-Go List:**

* **No Automated Meta Title Rewrites:** Never pipe this queue directly into a Generative AI tool to automatically rewrite and publish meta tags. The model cannot detect sensitive topics or nuanced brand voice.

* **No Automated Content Deletion:** Low scores represent a lack of current search performance, not necessarily a lack of business value. Pages must not be pruned without checking underlying revenue or direct-traffic metrics.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**When to Retrain or Suspend the Playbook:**
The recommendations generated by this playbook will go stale if the underlying search environment shifts.

**The model should be re-evaluated if any of the following occur:**

1. **Base Rate Shift:** If the baseline average of pages ranking on Page 1 with < 2% CTR shifts by more than 10% month-over-month.

2. **SERP Layout Changes:** If a major search engine update introduces new, dominant visual elements (like AI overviews or expanded shopping carousels) that permanently alter what a "normal" CTR looks like for a Page 1 ranking.

3. **Data Contract Break:** If the upstream warehouse changes how `gsc_impressions` or `gsc_clicks` are aggregated.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [1]:
# Load Dataset
import pandas as pd
from datasets import load_dataset
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

hf_token = userdata.get('HF_TOKEN')

print("Connecting to stream...")
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=hf_token
)

print("Streaming data to find February 2025 records...")
records = []
for i, row in enumerate(ds):
    if str(row.get('report_date', '')).startswith('2025-02'):
        records.append(row)
    if len(records) >= 10000:
        break

df_clean = pd.DataFrame(records)
df_clean['report_date'] = pd.to_datetime(df_clean['report_date'])

df_clean = df_clean.dropna(subset=['gsc_avg_position', 'gsc_impressions', 'gsc_clicks']).copy()
df_clean = df_clean[(df_clean['gsc_avg_position'] > 0) & (df_clean['gsc_impressions'] > 0)]
df_clean['ctr'] = (df_clean['gsc_clicks'] / df_clean['gsc_impressions']) * 100

df_clean['label_needs_fix'] = ((df_clean['gsc_avg_position'] <= 10) & (df_clean['ctr'] <= 2.0)).astype(int)
features = ['gsc_impressions', 'ga4_sessions', 'sessions_organic', 'sessions_direct']

X = df_clean[features].fillna(0)
y = df_clean['label_needs_fix']
groups = df_clean['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, y_train_grp = X.iloc[train_idx], y.iloc[train_idx]
rf_honest = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf_honest.fit(X_train_grp, y_train_grp)

print(f"Data loaded and honest model trained.")

Connecting to stream...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Streaming data to find February 2025 records...
Data loaded and honest model trained. Ready to export queue!


In [2]:
import os
import pandas as pd
import numpy as np

df_clean['fix_probability'] = rf_honest.predict_proba(X)[:, 1]

# Filter for actionable items
action_queue = df_clean[df_clean['fix_probability'] > 0.50].copy()

# Attach reason codes and actions
action_queue['reason_code'] = 'high_vis_poor_capture'
action_queue['recommended_action'] = 'Manual Meta Title Review'

# Sort by probability (highest confidence first), then by impressions (highest impact)
action_queue = action_queue.sort_values(
    by=['fix_probability', 'gsc_impressions'],
    ascending=[False, False]
)

# Select the clean columns needed for the deployed research paper
export_cols = [
    'content_hash_id', 'report_date', 'gsc_impressions', 'ctr',
    'fix_probability', 'reason_code', 'recommended_action'
]
final_export = action_queue[export_cols]

# Create the outputs directory if it doesn't exist
os.makedirs("work/outputs", exist_ok=True)

# Export to CSV for the paper to consume next week
export_path = "work/outputs/w07_action_playbook_queue.csv"
final_export.to_csv(export_path, index=False)

print(f"Action playbook exported successfully to: {export_path}")
print(f"Total actionable URLs flagged for human review: {len(final_export)}")

Action playbook exported successfully to: work/outputs/w07_action_playbook_queue.csv
Total actionable URLs flagged for human review: 1560


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.